In [1]:
# ================================
# Comprehensive Generalizable Anomaly Detection with Ablation Studies (GPU-enabled)
# ================================

# --- Imports ---
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import random
from typing import Dict, List, Tuple, Optional


# Set seed for reproducibility
np.random.seed(0)
torch.manual_seed(0)

if torch.cuda.is_available():
    print("GPU is available")

# Use GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# =========================================
# STEP 1: Load and preprocess data
# =========================================
df = pd.read_csv("/home/azureuser/cloudfiles/code/Users/hhomayouni/context-aware contrastive learning for anomaly detection and explanation/breast-cancer.csv")
df = df.drop(columns=["ID"])
y = df["Class"].values
X = df.drop(columns=["Class"])

print("Data Distribution:")
print(df.groupby("Class").size())

categorical_cols = X.select_dtypes(include="object").columns.tolist()
numeric_cols = X.select_dtypes(include="number").columns.tolist()

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler())
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_cols),
    ("cat", categorical_transformer, categorical_cols)
])

X_processed = preprocessor.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X_processed, y, test_size=0.2, stratify=y, random_state=42)
# Set anomaly threshold based on the ground truth
#anom_threshold= 100*sum(df["Class"]==1)/len(df)
anom_threshold=5

df

Data Distribution:
Class
0    444
1    239
dtype: int64


,CT,Ui,Uh,Mh,SS,BN,BC,NN,Mi,Class
0,5,1,1,1,0,1,3,1,1,0
1,5,1,1,5,7,10,3,0,1,0
2,3,1,1,1,0,0,3,1,1,0
3,6,8,8,1,3,1,3,7,1,0
4,1,1,1,3,0,1,3,1,1,0
...,...,...,...,...,...,...,...,...,...,...
678,3,1,1,1,3,0,1,1,1,0
679,0,1,1,1,0,1,1,1,1,0
680,5,10,10,3,7,3,8,10,0,1
681,1,8,6,1,3,1,10,6,1,1


In [2]:
# =========================================
# STEP 2: Helper Functions
# =========================================
def create_feature_partitions(X, K):
    """
    Create K semantic feature partitions {P₁, ..., P_K} using correlation-based clustering,
    as described in the CACL framework (Section 4.1 of the paper).

    Each feature f_j in the input dataset X ∈ ℝ^{N×D} is grouped into one of K disjoint
    subsets (partitions) based on its correlation structure with other features.
    These partitions are later used to construct multi-view semantic representations
    x = [x^{(1)}, ..., x^{(K)}], where x^{(k)} contains features in P_k.

    Method
    ------
    1. Compute Corr(Xᵀ) ∈ ℝ^{D×D}, the Pearson correlation matrix across D features.
    2. Apply PCA with K components to Corr(Xᵀ), yielding a reduced representation
       G ∈ ℝ^{D×K}, where row g_j is the K-dim embedding of feature f_j.
    3. Run KMeans clustering on the rows of G to assign each feature to a cluster.
    4. Return a list of K partitions, where each partition P_k is a list of feature indices.

    Parameters
    ----------
    X : np.ndarray of shape (N, D)
        Input data matrix with N samples (records) and D features.
    K : int
        Number of desired feature partitions (K ≥ 2).

    Returns
    -------
    partitions : List[List[int]]
        A list of K feature groups {P₁, ..., P_K}, where each P_k contains indices
        of features assigned to partition k.

    References
    ----------
    Section 4.1, Equation:
        G = PCA_K(Corr(Xᵀ)) ∈ ℝ^{D×K}
    Figure 4.1 shows this process and how it feeds into semantic view creation.

    Example
    -------
    >>> parts = create_feature_partitions(X, K=4)
    >>> print(len(parts))  # 4 partitions
    >>> print(sum(len(p) for p in parts)) == X.shape[1]  # all features assigned
    True
    """
    corr = np.corrcoef(X.T)
    reduced = PCA(n_components=K).fit_transform(corr)
    clusters = KMeans(n_clusters=K, random_state=0).fit(reduced)
    partitions = [[] for _ in range(K)]
    for i, label in enumerate(clusters.labels_):
        partitions[label].append(i)
    return partitions


def plot_training_loss(losses, title="Training Loss"):
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(losses, marker='o', label="Train Loss", color='black')
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.grid(True)
    ax.legend()
    fig.tight_layout()
    return fig


def _feature_to_partition_map(partitions: List[List[int]]) -> Dict[int, int]:
    """
    Construct a reverse mapping from feature index j to its assigned partition P_k,
    as used in the semantic view construction phase of CACL.

    Given:
        • A set of K disjoint feature partitions {P₁, ..., P_K}, where each P_k is a
          list of feature indices grouped together based on correlation structure.

    Returns:
        • A dictionary mapping each feature index f_j ∈ {0, ..., D−1} to its partition
          label k ∈ {0, ..., K−1} such that:
              f_j ∈ P_k  ⇒  map[f_j] = k

    Parameters
    ----------
    partitions : List[List[int]]
        A list of K disjoint groups of feature indices, representing {P₁, ..., P_K}.
        Each group contains feature indices assigned to that partition.

    Returns
    -------
    fmap : Dict[int, int]
        A dictionary such that fmap[f_j] = k where f_j ∈ P_k.

    Example
    -------
    >>> parts = [[0, 2, 5], [1, 3, 4]]
    >>> _feature_to_partition_map(parts)
    {0: 0, 2: 0, 5: 0, 1: 1, 3: 1, 4: 1}

    Related Notation (CACL paper)
    -----------------------------
    - P_k : The k-th semantic feature partition (subset of features)
    - x^{(k)} : The view of record x formed from features in P_k
    - (f_j ∈ P_k) ⇒ j maps to k
    """
    fmap = {}
    for k, feats in enumerate(partitions):
        for f in feats:
            fmap[int(f)] = int(k)
    return fmap


def build_context_groups(X, k: int = 5, mode: str = "static"):
    """
    Construct context groups G(i) for each record xᵢ ∈ ℝ^D, based on the modality.

    These context groups G(i) are used in the context-aware contrastive loss (L_ctx+mv)
    to model semantic similarity between records beyond intra-sample views (CACL Sections 2.1.2, 4.2).

    The method adapts the definition of context group G(i) to the data type:
      - "tabular"  : KMeans clustering followed by kNN *within cluster* (G(i) = semantically similar cluster members).
      - "spatial"  : standard kNN using Euclidean distance (G(i) = nearby points).
      - "temporal" : fixed-size window (G(i) = neighbors within ±k rows).

    Parameters
    ----------
    X : np.ndarray of shape (N, D)
        Dataset with N records (rows) and D features (columns).
    k : int
        Size of each context group G(i). For temporal, this is the half-window size.
    mode : str
        One of {"tabular", "spatial", "temporal"}.
        Default "static" is treated as "tabular" for backward compatibility.

    Returns
    -------
    ctx : List[List[int]]
        A list of N context groups: ctx[i] = G(i) = indices of semantically similar neighbors of xᵢ.

    CACL Notation Reference
    -----------------------
    - G(i) : context group for record i (used in P_ctx in Equation for L_ctx+mv):contentReference[oaicite:1]{index=1}
    - P_ctx = { h_j^{(k)} | j ∈ G(i) } — cross-record positives in contrastive loss
    - Section 2.1.2: "... G(i) may be defined as kNN(xᵢ) for spatial data, sliding window for temporal data,
      or learned clusters for static/tabular data.":contentReference[oaicite:2]{index=2}

    Mode Details
    ------------
    - tabular  : cluster in feature space using KMeans → G(i) = kNN within cluster.
    - spatial  : G(i) = Euclidean kNN over entire dataset.
    - temporal : G(i) = fixed-size neighborhood window excluding self.

    Examples
    --------
    >>> ctx = build_context_groups(X, k=5, mode="tabular")
    >>> len(ctx) == X.shape[0]
    True
    >>> all(i not in ctx[i] for i in range(len(ctx)))  # self excluded
    True
    """
    N = len(X)
    if N == 0:
        return []

    mode_l = (mode or "tabular").lower()
    if mode_l == "static":
        mode_l = "tabular"

    # ---------- TEMPORAL: fixed window neighbors ----------
    if mode_l == "temporal":
        k_win = int(max(1, min(k, max(1, N - 1))))
        ctx = []
        for i in range(N):
            left = list(range(max(0, i - k_win), i))
            right = list(range(i + 1, min(N, i + k_win + 1)))
            ctx.append(left + right)
        return ctx

    # Helper: compute kNN neighbors (excluding self) with Euclidean metric
    def _knn_neighbors(X_arr, k_neighbors):
        from sklearn.neighbors import NearestNeighbors
        k_eff = int(min(max(1, k_neighbors) + 1, len(X_arr)))  # +1 for self
        nbrs = NearestNeighbors(n_neighbors=k_eff, metric="euclidean").fit(X_arr)
        _, inds = nbrs.kneighbors(X_arr)  # includes self as first index
        out = []
        for row in inds:
            # drop self (the first is guaranteed to be self when k_eff>=1)
            out.append([int(j) for j in row if j != row[0]][:k_neighbors])
        return out

    X_arr = np.asarray(X)

    # ---------- SPATIAL: plain kNN over all points ----------
    if mode_l == "spatial":
        return _knn_neighbors(X_arr, k_neighbors=int(max(1, min(k, N - 1))))

    # ---------- TABULAR: cluster, then nearest inside cluster ----------
    if mode_l == "tabular":
        try:
            from sklearn.cluster import KMeans
            # Heuristic cluster count: ~sqrt(N), at least 2 and at most N
            n_clusters = int(max(2, min(N, round(np.sqrt(N)))))
            # n_init API differs across sklearn versions; handle both
            try:
                km = KMeans(n_clusters=n_clusters, random_state=0, n_init="auto")
            except TypeError:
                km = KMeans(n_clusters=n_clusters, random_state=0, n_init=10)
            labels = km.fit_predict(X_arr)
        except Exception:
            # If sklearn not available or fails, fall back to global kNN
            return _knn_neighbors(X_arr, k_neighbors=int(max(1, min(k, N - 1))))

        # Pre-index members for each cluster
        from collections import defaultdict
        cluster_members = defaultdict(list)
        for idx, lab in enumerate(labels):
            cluster_members[int(lab)].append(idx)

        # For each point, take up to k nearest neighbors *within its cluster*.
        # If the cluster is too small, fill the remainder with global kNN.
        ctx = [[] for _ in range(N)]

        # Precompute global kNN once for fallback filling
        global_knn = _knn_neighbors(X_arr, k_neighbors=int(max(1, min(k, N - 1))))

        for i in range(N):
            lab = int(labels[i])
            members = cluster_members[lab]
            # Candidates are all points in same cluster except i
            cand = [j for j in members if j != i]
            if len(cand) == 0:
                ctx[i] = global_knn[i][:k]
                continue

            # Sort candidates by Euclidean distance to i
            diffs = X_arr[cand] - X_arr[i]
            dists = np.einsum("nd,nd->n", diffs, diffs)  # squared distances
            order = np.argsort(dists)
            local_neighbors = [int(cand[o]) for o in order[:k]]

            # If not enough neighbors in cluster, pad with global kNN (excluding duplicates)
            if len(local_neighbors) < k:
                extras = [j for j in global_knn[i] if j not in local_neighbors and j != i]
                need = k - len(local_neighbors)
                local_neighbors += extras[:need]

            ctx[i] = local_neighbors[:k]

        return ctx

    # ---------- Unknown mode: default to global kNN ----------
    return _knn_neighbors(X_arr, k_neighbors=int(max(1, min(k, N - 1))))



def _pick_threshold(scores: np.ndarray,
                    y_true: np.ndarray,
                    normal_percentile: float = 95.0):
    """
    Choose a threshold robustly.
    1) Try the given percentile on NORMALS (controls FPR).
    2) If that produces all-zeros or all-ones predictions, fall back to an F1-optimal threshold
       searched over score quantiles.
    Returns: threshold (float), preds (0/1 np.ndarray)
    """
    scores = np.asarray(scores, dtype=float)
    y_true = np.asarray(y_true, dtype=int)
    neg = (y_true == 0)

    # (1) normal-quantile threshold (if we have normals)
    if neg.any():
        thr = float(np.nanpercentile(scores[neg], normal_percentile))
    else:
        thr = float(np.nanpercentile(scores, normal_percentile))

    preds = (scores >= thr).astype(int)

    # If degenerate (all 0s or all 1s), search a better threshold for F1
    if preds.sum() == 0 or preds.sum() == len(preds):
        qs = np.linspace(50, 99.5, 100)  # try a range of high quantiles
        best_f1, best_thr = -1.0, thr
        for q in qs:
            t = float(np.nanpercentile(scores, q))
            p = (scores >= t).astype(int)
            # avoid divisions-by-zero safely
            tp = int(((p == 1) & (y_true == 1)).sum())
            fp = int(((p == 1) & (y_true == 0)).sum())
            fn = int(((p == 0) & (y_true == 1)).sum())
            prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
            rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
            f1   = (2*prec*rec)/(prec+rec) if (prec+rec) > 0 else 0.0
            if f1 > best_f1:
                best_f1, best_thr = f1, t
        thr = best_thr
        preds = (scores >= thr).astype(int)

    return thr, preds


In [3]:
# =========================================
# STEP 3: Dataset & Model Classes (MLP and Transformer)
# =========================================

class PartitionedDataset(Dataset):
    """
    Torch dataset for multi-view contrastive training under CACL.

    Each record x ∈ ℝ^D is partitioned into K views {x^{(1)}, ..., x^{(K)}}
    using feature partitions {P₁, ..., P_K} such that x^{(k)} = x[P_k].

    Parameters
    ----------
    X : np.ndarray of shape (N, D)
        Full dataset with N samples (rows) and D features (columns).
    partitions : List[List[int]]
        A list of K disjoint feature index groups, typically created by
        `create_feature_partitions`. Each partition P_k determines one
        semantic view x^{(k)} used in contrastive training.

    Returns
    -------
    __getitem__(i) → (views, i)
        views: tuple of K tensors, each of shape (|P_k|,)
        i:     index of the sample (used to construct context-aware loss)
    """
    def __init__(self, X, partitions):
        self.X = X
        self.partitions = partitions

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = self.X[idx]
        views = tuple(torch.tensor(x[part], dtype=torch.float32) for part in self.partitions)
        return views, idx

def collate_fn(batch):
    """
    Collate function for multi-view batches.

    Transforms a batch of (views, idx) into:
      - List of K tensors [B, |P_k|] → one tensor per view.
      - List of sample indices (length B), preserved for context-aware contrastive loss.

    Used in DataLoader to batch samples with partitioned views.
    """
    views_batch, indices = zip(*batch)
    num_parts = len(views_batch[0])
    batch_views = [torch.stack([v[i] for v in views_batch]) for i in range(num_parts)]
    return batch_views, list(indices)

# ---- MLP Encoder ----
class MLPEncoder(nn.Module):
    """
    Partition encoder f_θ for tabular views (Equation for f^{(k)} in the paper).

    A shallow two-layer MLP that maps x^{(k)} ∈ ℝ^{|P_k|} → z^{(k)} ∈ ℝ^d.
    """
    def __init__(self, input_dim, hidden=64, output=64):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(input_dim, hidden), nn.ReLU(), nn.Linear(hidden, output))

    def forward(self, x):
        return self.net(x)

# ---- Transformer Encoder ----
class TransformerEncoder(nn.Module):
    """
    Alternative encoder f_θ for structured/tabular data using Transformer blocks
    (also noted in Section 4.3).

    Each x^{(k)} is treated as a short sequence and encoded via a shared encoder.
    Projects input features to output_dim, applies TransformerEncoder.

    Parameters
    ----------
    input_dim : int
        Number of features in the partition P_k.
    output_dim : int
        Dimensionality of the encoded output h^{(k)}.
    nhead : int
        Number of attention heads.
    num_layers : int
        Number of stacked Transformer encoder layers.
    """
    def __init__(self, input_dim, output_dim=64, nhead=2, num_layers=1):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, output_dim)
        encoder_layer = nn.TransformerEncoderLayer(d_model=output_dim, nhead=nhead)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

    def forward(self, x):
        # Input: [B, D] -> [1, B, D] (treat features as sequence)
        x = self.input_proj(x).unsqueeze(0)
        out = self.transformer(x)  # [1, B, D]
        return out.squeeze(0)

# ---- Projector ----
class Projector(nn.Module):
    """
    Projection head g_ϕ for latent embeddings.

    Maps z^{(k)} to h^{(k)} ∈ ℝ^d and applies L2 normalization
    to produce unit-norm embeddings for cosine-based contrastive learning.
    """
    def __init__(self, input_dim=64, proj_dim=32):
        super().__init__()
        self.fc = nn.Linear(input_dim, proj_dim)

    def forward(self, x):
        return F.normalize(self.fc(x), dim=-1)

# ---- MultiViewModel with selectable encoder ----
class MultiViewModel(nn.Module):
    """
    Multi-partition encoder and projector for CACL.

    Given K views {x^{(1)}, ..., x^{(K)}}, this model applies a shared
    encoder f_θ^{(k)} followed by a projection head g_ϕ^{(k)} to produce
    embeddings h^{(k)} used in contrastive training and anomaly detection.

    Parameters
    ----------
    partitions : List[List[int]]
        Feature groups {P₁, ..., P_K} used to generate views x^{(k)}.
    encoder_type : str, "mlp" or "transformer"
        Selects the encoder backbone f_θ: shallow MLP or Transformer.

    Returns
    -------
    forward(views: List[Tensor]) → List[Tensor]
        Input: List of K tensors [B, |P_k|] for each view.
        Output: List of K tensors [B, d] — L2-normalized latent embeddings {h^{(k)}}.
    """
 
    def __init__(self, partitions, encoder_type="mlp"):
        super().__init__()
        self.encoder_type = encoder_type
        if encoder_type == "mlp":
            self.encoders = nn.ModuleList([MLPEncoder(len(p)) for p in partitions])
        elif encoder_type == "transformer":
            self.encoders = nn.ModuleList([TransformerEncoder(len(p)) for p in partitions])
        else:
            raise ValueError("Unsupported encoder_type. Choose 'mlp' or 'transformer'.")
        self.projectors = nn.ModuleList([Projector() for _ in partitions])

    def forward(self, views):
        return [self.projectors[i](self.encoders[i](views[i])) for i in range(len(views))]

In [4]:
# =========================================
# STEP 4: Define Loss Function
# =========================================

# =========================================
# Original Contrastive Loss
# =========================================
def contrastive_loss(embeddings, tau=0.1):
    """
    Multi-view InfoNCE loss (L_mv) for intra-record contrastive learning.

    Each sample x ∈ ℝ^D is partitioned into K semantic views: x = [x^{(1)}, ..., x^{(K)}].
    For each partition embedding h^{(k)} = g_ϕ(f_θ(x^{(k)})), we treat other views from the
    same record as positives (P_mv), and all views from other records as negatives (N).

    The loss encourages agreement between partitions of the same record while pushing
    apart all views from different records.

    Parameters
    ----------
    embeddings : List[Tensor], length K
        Each tensor has shape (B, d), where B is batch size and d is embedding dim.
        embeddings[k][i] = h_i^{(k)} — the embedding of partition k for record i.
    tau : float
        Temperature scaling parameter for cosine similarities.

    Returns
    -------
    loss : torch.Tensor
        Scalar contrastive loss averaged over B × K anchor views.

    References
    ----------
    Equation from CACL paper:
        L_mv = −(1/BK) ∑_{i=1}^B ∑_{k=1}^K log [ ∑_{l≠k} exp(sim(h_i^{(k)}, h_i^{(l)}) / τ)
                                            / ∑_{j≠i,l} exp(sim(h_i^{(k)}, h_j^{(l)}) / τ) ]
    """
    B, K = embeddings[0].shape[0], len(embeddings)
    all_views = torch.stack(embeddings, dim=1)
    loss = 0.0
    for i in range(B):
        for k in range(K):
            anchor = all_views[i, k]
            pos = [all_views[i, l] for l in range(K) if l != k]
            sim_pos = torch.stack([
                torch.exp(F.cosine_similarity(anchor, p, dim=0) / tau)
                for p in pos]).sum()
            neg = [all_views[j, l] for j in range(B) if j != i for l in range(K)]
            sim_neg = torch.stack([
                torch.exp(F.cosine_similarity(anchor, n, dim=0) / tau)
                for n in neg]).sum()
            loss -= torch.log(sim_pos / (sim_pos + sim_neg + 1e-8))
    return loss / (B * K)


# =========================================
# Context-Aware Contrastive Loss
# =========================================
def contrastive_loss_ctx(embeddings, ctx_idx, batch_idx,
                         alpha_mv=1.0, alpha_ctx=0.5, tau=0.1, max_ctx_samples=3):
    """
    Context-aware contrastive loss with safe fallbacks:
      - If no negatives remain after excluding context, relax to all non-anchor examples.
      - Positives include (i) other views of the same sample and (ii) a few context neighbors.
    """
    import random
    B, K = embeddings[0].shape[0], len(embeddings)
    all_views = torch.stack(embeddings, dim=1)  # (B,K,d)
    loss = 0.0

    # Fast path: empty ctx_idx protection
    has_ctx = ctx_idx is not None

    for i in range(B):
        anchor_gid = int(batch_idx[i])
        neigh_set = set(ctx_idx[anchor_gid]) if has_ctx else set()
        anchor_views = all_views[i]  # (K,d)

        for k in range(K):
            anchor = anchor_views[k]

            # (1) multi-view positives from same sample
            pos = [anchor_views[l] for l in range(K) if l != k]
            weights = [torch.tensor(alpha_mv, device=anchor.device)] * len(pos)

            # (2) a few context positives (same partition k) from neighbor samples
            if has_ctx:
                ctx_cands = [j for j, gid in enumerate(batch_idx) if gid in neigh_set and gid != anchor_gid]
                if max_ctx_samples is not None and max_ctx_samples > 0:
                    ctx_cands = random.sample(ctx_cands, min(max_ctx_samples, len(ctx_cands)))
                for j in ctx_cands:
                    pos.append(all_views[j, k])
                    weights.append(torch.tensor(alpha_ctx, device=anchor.device))

            sim_pos = torch.stack([
                w * torch.exp(F.cosine_similarity(anchor, p, dim=0) / tau)
                for p, w in zip(pos, weights)
            ]).sum()

            # Negatives: all non-anchor, excluding context; fallback if empty
            neg = [all_views[j, l]
                   for j, gid in enumerate(batch_idx)
                   if gid != anchor_gid and (not has_ctx or gid not in neigh_set)
                   for l in range(K)]
            if len(neg) == 0:
                # relax exclusion: use all non-anchor examples
                neg = [all_views[j, l] for j in range(B) if j != i for l in range(K)]

            sim_neg = torch.stack([
                torch.exp(F.cosine_similarity(anchor, n, dim=0) / tau)
                for n in neg
            ]).sum()

            loss = loss - torch.log(sim_pos / (sim_pos + sim_neg + 1e-8))

    return loss / (B * K)


In [5]:
# =========================================
# Step 5: Explanation Modules
# =========================================

#Q. How about record-level anomalies: the paper: For record-level anomalies, 
#the framework also considers the context group \(\mathcal{G}(x)\) associated with the flagged record. 
#Each member of the group is compared to the flagged record via its average embedding:
#h^{(\text{avg})}(x) = \frac{1}{K} \sum_{k=1}^{K} h^{(k)}, \quad
#d_{x,x'} = 1 - \text{sim}\left(h^{(\text{avg})}(x), h^{(\text{avg})}(x')\right), \quad x' \in \mathcal{G}(x).
#The maximum \(d_{x,x'}\) or the average context deviation indicates whether the record is misaligned with its peers.
#This step thus localizes the most salient intra-record or context-level conflict in the flagged sample.

match_table = []  

def _upper_triangle_argmax(mat: np.ndarray) -> Tuple[int, int]:
    """
    Identify the pair of feature partitions (k⋆, l⋆) with the highest disagreement
    in the upper triangle of a symmetric disagreement matrix D ∈ ℝ^{K×K}.

    This is used in CACL's explanation module to extract the
    most semantically inconsistent partition pair within a flagged record:
        (k⋆, l⋆) = argmax_{k < l} D_{k,l}

    The function ensures that:
      • The diagonal D_{k,k} is ignored (self-similarity is not considered).
      • The matrix is assumed symmetric (D_{k,l} = D_{l,k}), so only the upper
        triangle (k < l) is evaluated to avoid duplicate comparisons.
      • If all entries are NaN or -inf (e.g., empty matrix), falls back to (0,1).

    Parameters
    ----------
    mat : np.ndarray of shape (K, K)
        A symmetric square matrix of pairwise disagreement values D_{k,l} between
        partition embeddings h^{(k)} and h^{(l)}.
        Typically D_{k,l} = 1 − cosine(h^{(k)}, h^{(l)}).

    Returns
    -------
    (k_star, l_star) : Tuple[int, int]
        Indices of the partition pair (k⋆, l⋆) with the maximum disagreement value
        in the strict upper triangle (k < l).

    CACL Notation Reference
    -----------------------
    • D_{k,l} : disagreement between partition k and l
    • (k⋆, l⋆) = argmax_{k < l} D_{k,l} — the most conflicting pair

    Example
    -------
    >>> D = np.array([
    ...     [0.0, 0.4, 0.1],
    ...     [0.4, 0.0, 0.7],
    ...     [0.1, 0.7, 0.0]
    ... ])
    >>> _upper_triangle_argmax(D)
    (1, 2)  # since D[1,2] = 0.7 is the max in upper triangle
    """
    assert mat.ndim == 2 and mat.shape[0] == mat.shape[1], "Expected square matrix"
    # Set diagonal and lower triangle to -inf so they are ignored
    tri = np.triu(mat, k=1)
    # Handle all -inf case (fallback to (0,1))
    if not np.isfinite(tri).any():
        return (0, 1) if mat.shape[0] > 1 else (0, 0)
    idx = np.nanargmax(tri)  # nan-safe
    k_star, l_star = np.unravel_index(idx, tri.shape)
    return int(k_star), int(l_star)

# Q:
'''
In parallel, to explain record-level inconsistencies, the framework tracks the empirical distribution of context alignment scores:
$
a_i = \frac{1}{|\mathcal{G}(x_i)|} \sum_{x_j \in \mathcal{G}(x_i)} \text{sim}\left(h^{(\text{avg})}(x_i), h^{(\text{avg})}(x_j)\right),
$
where \(x_i\) ranges over inlier records. If a flagged record \(x\) exhibits a context similarity score \(a_x\) significantly below the training distribution (e.g., below a predefined quantile), the system reports that the record violates typical group-level consistency:
\emph{“This record is inconsistent with others in its context group.”}
'''

def compute_dependency_matrix(X_inliers, model, partitions, eta=0.2):
    """
    Estimate the cross-partition semantic dependency matrix P ∈ [0,1]^{K×K}
    using only inlier (normal: The training set, if we trust it to be mostly clean) records and their embeddings.

    This matrix encodes how frequently each pair of partitions (P_k, P_l) agrees
    across the inlier distribution, and is used during explanation to detect
    violated semantic dependencies (CACL Section 5.1, Step 2):contentReference[oaicite:1]{index=1}.

    Conceptually:
      - For each record x ∈ X_inliers, we embed each semantic view x^{(k)} using the trained encoder:
            h^{(k)} = g_ϕ(f_θ(x^{(k)})) ∈ ℝ^d
      - We compute the pairwise disagreement matrix:
            D_{k,l}(x) = 1 − cosine(h^{(k)}, h^{(l)})
      - A partition pair (k, l) is said to "agree" if D_{k,l}(x) < η.
      - Across all inliers, we compute:
            P_{k,l} = Pr[D_{k,l}(x) < η] ≈ (1/N) ∑_i 1{ D_{k,l}(x_i) < η }

    Parameters
    ----------
    X_inliers : np.ndarray of shape (N, D)
        A collection of inlier (non-anomalous) records.
        These should represent clean data that preserves normal cross-partition dependencies.
        In practice, this is typically drawn from:
          - The training set (if assumed clean), or
          - A curated validation set of "normal" records.
    model : MultiViewModel
        Trained model implementing:
            model(views: List[Tensor]) → List[Tensor]
        where each view corresponds to a semantic partition.
    partitions : List[List[int]]
        A list of K feature index groups {P₁, ..., P_K}, used to slice views x^{(k)}.
    eta : float, default=0.2
        Cosine distance threshold below which two partitions are considered
        semantically consistent (i.e., D_{k,l}(x) < η ⇒ agreement).

    Returns
    -------
    P : np.ndarray of shape (K, K)
        A symmetric matrix of semantic agreement probabilities:
            P[k, l] ≈ Pr_{x ∈ inliers}[ D_{k,l}(x) < η ]
        This captures how often each pair of partitions agrees in **embedding space**.

    Notes
    -----
    - Computation is done in **embedding space**, not raw feature space.
    - Used to detect **violated dependencies** during explanation:
          If P[k,l] > ρ (normally consistent) but D_{k,l}(x_test) > η (inconsistent),
          then (k,l) is flagged as a violated rule:contentReference[oaicite:2]{index=2}.
    - Related Equation (CACL Section 5.1):
          P_{k,l} = (1/N) ∑_i 1{ D_{k,l}(x_i) < η }

    Example
    -------
    >>> P = compute_dependency_matrix(X_train, model, partitions)
    >>> assert P.shape == (K, K)
    >>> np.all(P <= 1.0) and np.all(P >= 0.0)
    True
    """
    N = len(X_inliers)
    K = len(partitions)
    Lambda_sum = np.zeros((K, K))

    model.eval()
    with torch.no_grad():
        for i in range(N):
            views = [torch.tensor(X_inliers[i][p], dtype=torch.float32).unsqueeze(0).to(model.device)
                    for p in partitions]
            embs = model(views)
            embeddings = [e.squeeze(0).cpu().numpy() for e in embs]
            D = np.zeros((K, K))
            for a in range(K):
                for b in range(K):
                    if a != b:
                        D[a, b] = 1 - F.cosine_similarity(
                            torch.tensor(embeddings[a]),
                            torch.tensor(embeddings[b]),
                            dim=0
                        ).item()
            Lambda = (D < eta).astype(int)
            Lambda_sum += Lambda

    return Lambda_sum / N


def compute_context_embeddings(model, X, partitions):
    """
    Compute per-record average embedding h̄(x) across all K semantic partitions of each record.

    For each record x ∈ ℝ^D, this function:
      - Splits x into K semantic views: x = [x^{(1)}, ..., x^{(K)}]
      - Computes partition embeddings h^{(k)} = g_ϕ(f_θ(x^{(k)}))
      - Averages the embeddings to produce h̄(x) = (1/K) ∑ h^{(k)}

    This is used to define context embeddings for evaluating context similarity
    sim_ctx(x, G(x′)) in CACL.

    Parameters
    ----------
    model : MultiViewModel
        Trained model with partition-wise encoders and projectors.
    X : np.ndarray of shape (N, D)
        Dataset to compute embeddings for (e.g., X_test or X_mutated).
    partitions : List[List[int]]
        Feature partitions {P₁, ..., P_K}, as generated by create_feature_partitions.

    Returns
    -------
    ctx_embeddings : np.ndarray of shape (N, d)
        Average embeddings for each record across its K views.

    Example
    -------
    >>> ctx_embeddings = compute_context_embeddings(model, X_test, partitions)
    >>> ctx_embeddings.shape  # (N, d)
    """
    import numpy as np
    import torch

    model.eval()
    device = getattr(model, "device", torch.device("cuda" if torch.cuda.is_available() else "cpu"))
    N = len(X)
    K = len(partitions)

    embeddings = []

    with torch.no_grad():
        for i in range(N):
            views = [
                torch.tensor(X[i, part], dtype=torch.float32).unsqueeze(0).to(device)
                for part in partitions
            ]
            out = model(views)  # List of (1, d)
            vecs = [e.squeeze(0).cpu().numpy() for e in out]
            avg = np.mean(np.stack(vecs, axis=0), axis=0)  # (d,)
            embeddings.append(avg)

    return np.vstack(embeddings)  # shape (N, d)



def explain_record(
    record_embeddings: List[np.ndarray],
    context_avg_embeddings: Optional[np.ndarray] = None,
    context_indices: Optional[List[int]] = None,
    eta: float = 0.2,
    dependency_matrix: Optional[np.ndarray] = None,
    context_threshold_quantile: Optional[float] = None,
    ref_ctx_sim_distribution: Optional[np.ndarray] = None,
) -> Dict:
    """
    Generate a structured explanation for a single record x based on partition-level
    semantic disagreement and contextual misalignment.

    This function implements the 3-part explanation method described in the CACL paper, returning:
      - The most semantically conflicting partition pair (k⋆, l⋆)
      - Any violated statistical dependencies (based on P_{k,l} from training)
      - Context misalignment (if context groups are defined)

    Inputs
    ------
    record_embeddings : List[np.ndarray] of length K
        Embeddings {h^{(1)}, ..., h^{(K)}} for the K partitions of a test record x.
        Each h^{(k)} ∈ ℝ^d is typically normalized.
    context_avg_embeddings : np.ndarray, shape (N, d), optional
        Average embeddings of other records used for context alignment (e.g., {h̄(x′)}).
        Required only for evaluating context similarity.
    context_indices : List[int], optional
        Indices G(x) of context neighbors for the current record (same index space as context_avg_embeddings).
    eta : float, default=0.2
        Threshold for semantic agreement: D_{k,l} < η means partition k and l are "consistent".
    dependency_matrix : np.ndarray, shape (K, K), optional
        Dependency matrix P ∈ [0,1]^{K×K} computed over inliers.
        Each entry P_{k,l} = Pr[D_{k,l} < η] in normal data.
        Used to flag violated dependencies at test time.
    context_threshold_quantile : float, optional
        Quantile threshold (e.g., 0.1) to compare context similarity against.
    ref_ctx_sim_distribution : np.ndarray, optional
        Distribution of context similarity scores from inliers, used to compute the threshold
        for flagging context violations.

    Returns
    -------
    explanation : dict with keys:
        - disagreement_matrix : np.ndarray (K×K), D_{k,l} = 1 − sim(h^{(k)}, h^{(l)})
        - most_conflicting_partitions : Tuple[int, int], (k⋆, l⋆) = argmax_{k<l} D_{k,l}
        - max_disagreement : float, value of D_{k⋆, l⋆}
        - violated_dependencies : List[Tuple[int, int]], pairs where P_{k,l} > ρ and D_{k,l}(x) > η
        - violated_dependencies_features : List, empty (reserved for fine-grained versions)
        - context_similarity : float or None, mean cosine similarity to neighbors
        - context_flag : bool, True if context similarity is below threshold

    Notes
    -----
    This method corresponds to:
      - Step 1: Identify most conflicting partition pair via D_{k,l}(x):contentReference[oaicite:1]{index=1}
      - Step 2: Flag violated dependencies where D_{k,l} > η but P_{k,l} > ρ:contentReference[oaicite:2]{index=2}
      - Step 3: Compute sim_ctx(x) = mean_cosine(h̄(x), h̄(x′)) over G(x):contentReference[oaicite:3]{index=3}
                If sim_ctx(x) < threshold, report context violation.

    """
    def _cos_sim(a, b):
        na = np.linalg.norm(a) + 1e-12
        nb = np.linalg.norm(b) + 1e-12
        return float(np.dot(a, b) / (na * nb))

    K = len(record_embeddings)
    H = np.vstack(record_embeddings)  # (K, d)
    # Pairwise disagreement
    D = np.zeros((K, K), dtype=float)
    for i in range(K):
        for j in range(K):
            D[i, j] = 1.0 - _cos_sim(H[i], H[j])
    k_star, l_star = _upper_triangle_argmax(D)
    max_disagreement = float(D[k_star, l_star])

    violated_deps = []
    violated_deps_features = []  # keep feature-level optional list empty for patched metric
    if dependency_matrix is not None:
        # dependency_matrix is Pkl (probability partitions agree) in [0,1]
        Kdm = dependency_matrix.shape[0]
        if Kdm != K:
            # Attempt to resize or skip if incompatible
            pass
        else:
            # For each pair where Pkl > rho (we infer rho ~ 0.9 by default if matrix looks probabilistic)
            # and current Dkl > eta => violated
            # If caller wishes, they can prefilter by rho and pass masked matrix.
            rho = 0.9
            for i in range(K):
                for j in range(i+1, K):
                    if dependency_matrix[i, j] > rho and D[i, j] > eta:
                        violated_deps.append((i, j))

    # Context similarity (optional)
    context_similarity = None
    ctx_flag = False
    if context_avg_embeddings is not None and context_indices:
        # Average embedding of record
        avg = H.mean(axis=0)
        # Compare to each neighbor's average embedding
        sims = []
        for j in context_indices:
            # context_avg_embeddings expected to be aligned with same index space as record's dataset
            sims.append(_cos_sim(avg, context_avg_embeddings[j]))
        if sims:
            context_similarity = float(np.mean(sims))
            if context_threshold_quantile is not None and ref_ctx_sim_distribution is not None:
                thr = float(np.quantile(ref_ctx_sim_distribution, context_threshold_quantile))
                ctx_flag = context_similarity < thr

    return {
        "disagreement_matrix": D,
        "most_conflicting_partitions": (k_star, l_star),
        "max_disagreement": max_disagreement,
        "violated_dependencies": violated_deps,
        "violated_dependencies_features": violated_deps_features,
        "context_similarity": context_similarity,
        "context_flag": ctx_flag,
    }

In [6]:
# Testing the explain record function

'''
# model            # the trained MultiViewModel
# partitions       # List[List[int]] feature partitions from create_feature_partitions
# X_test           # ndarray shape (N, D)
# dependency_matrix  # output from compute_dependency_matrix(...)
# ctx_embeddings   # ndarray shape (N, d), avg embeddings from unmutated X_test
# ctx_idx          # context group indices: List[List[int]], from build_context_groups

# --- Pick a test record ---
i = 2
x = X_test[i]  # shape (D,)

# ---------- Partitions ----------
partitions = create_feature_partitions(X_train, K=2)

# ---------- TRAIN context groups (for training only) ----------
k_ctx=3
try:
    ctx_idx_train = build_context_groups(X_train, k=k_ctx, mode="tabular")
except TypeError:
    ctx_idx_train = build_context_groups(X_train, k=k_ctx)

try:
    ctx_idx_test = build_context_groups(X_test, k=k_ctx, mode="tabular")
except TypeError:
    ctx_idx_test = build_context_groups(X_test, k=k_ctx)



# ---------- Model / Optim ----------
model = MultiViewModel(partitions, encoder_type="mlp").to(device)
model.device = device
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
dataset = PartitionedDataset(X_train, partitions)
loader = DataLoader(dataset, batch_size=128, shuffle=True, collate_fn=collate_fn)

# ---------- Train ----------
losses = []
model.train()
for epoch in range(5):
    epoch_losses = []
    for views, batch_idx in loader:
        views = [v.to(device) for v in views]
        embeddings = model(views)
        loss = contrastive_loss_ctx(embeddings, ctx_idx_train, batch_idx)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_losses.append(loss.item())
    avg_loss = float(np.mean(epoch_losses)) if epoch_losses else 0.0
    losses.append(avg_loss)
    print(f"Epoch {epoch+1}/{5} - Avg Loss: {avg_loss:.4f}")



# --- Get partition embeddings for the record ---
views = [
    torch.tensor(x[p], dtype=torch.float32).unsqueeze(0).to(model.device)
    for p in partitions
]
with torch.no_grad():
    embs = model(views)  # List of (1, d)
record_embeddings = [e.squeeze(0).cpu().numpy() for e in embs]  # List of (d,)

# --- Get context info (optional but recommended) ---
context_indices = ctx_idx_test[i] if ctx_idx_test is not None and i < len(ctx_idx_test) else []

dependency_matrix = compute_dependency_matrix(X_train, model, partitions, eta=0.2)

ctx_embeddings = compute_context_embeddings(model, X_test, partitions)

# Optionally compute context similarity reference quantile
ref_ctx_sim_distribution = None
if ctx_embeddings is not None:
    # Compute pairwise mean similarity to neighbors for a small inlier set
    sims = []
    for j in range(min(100, len(X_test))):
        neigh = ctx_idx_test[j] if j < len(ctx_idx_test) else []
        if not neigh: continue
        avg = ctx_embeddings[j]
        neighbor_sims = [
            np.dot(avg, ctx_embeddings[n]) / (
                np.linalg.norm(avg) * np.linalg.norm(ctx_embeddings[n]) + 1e-12)
            for n in neigh if n < len(ctx_embeddings)
        ]
        if neighbor_sims:
            sims.append(np.mean(neighbor_sims))
    ref_ctx_sim_distribution = np.array(sims)
# --- Call explain_record ---
explanation = explain_record(
    record_embeddings=record_embeddings,
    context_avg_embeddings=ctx_embeddings,
    context_indices=context_indices,
    eta=0.1,
    dependency_matrix=dependency_matrix,
    context_threshold_quantile=0.1,
    ref_ctx_sim_distribution=ref_ctx_sim_distribution
)

# --- Print result ---
from pprint import pprint
pprint(explanation)
'''

'\n# model            # the trained MultiViewModel\n# partitions       # List[List[int]] feature partitions from create_feature_partitions\n# X_test           # ndarray shape (N, D)\n# dependency_matrix  # output from compute_dependency_matrix(...)\n# ctx_embeddings   # ndarray shape (N, d), avg embeddings from unmutated X_test\n# ctx_idx          # context group indices: List[List[int]], from build_context_groups\n\n# --- Pick a test record ---\ni = 2\nx = X_test[i]  # shape (D,)\n\n# ---------- Partitions ----------\npartitions = create_feature_partitions(X_train, K=2)\n\n# ---------- TRAIN context groups (for training only) ----------\nk_ctx=3\ntry:\n    ctx_idx_train = build_context_groups(X_train, k=k_ctx, mode="tabular")\nexcept TypeError:\n    ctx_idx_train = build_context_groups(X_train, k=k_ctx)\n\ntry:\n    ctx_idx_test = build_context_groups(X_test, k=k_ctx, mode="tabular")\nexcept TypeError:\n    ctx_idx_test = build_context_groups(X_test, k=k_ctx)\n\n\n\n# ---------- Model

In [7]:
# =========================================
# Step 6: Evaluation via Mutated Anomalies
# =========================================

def mutate_test_set(
    X_test: np.ndarray,
    partitions=None,
    mutation_rate: float = 0.10,
    seed: int = 0,
    a4_noise_std: float = 0.02,
    ensure_all_types: bool = True,
    *,
    data_type: str = "tabular",     # {"tabular","spatial","temporal"}
    k_ctx: int = 5,                 # used for spatial A3 (kNN)
    n_clusters: int | None = None,  # used for tabular A3 (KMeans); default ~sqrt(N)
    window: int | None = None,      # used for temporal A3; default ~10% of N
    extreme_mult: float = 0.75      # how far beyond observed range to push A1/A2
) -> tuple[np.ndarray, dict[int, object]]:
    """
    Inject **synthetic anomalies** into X_test for mutation-based evaluation.

    We create a mutated copy `X_mut` and a metadata map `metas` that records the ground-truth
    *explanation target* for each mutated row:

        • **A1 (feature-level corruption)**: pick one feature f and push its value **beyond** the
          observed range (to hi + m·span or lo − m·span).  Meta → (f, f).
        • **A2 (cross-feature conflict)**: pick two features f1≠f2 and push them to **opposite**
          extremes beyond the range to maximize inconsistency.  Meta → (f1, f2).
        • **A3 (contextual inconsistency)**:
              – data_type="tabular": KMeans clusters; replace row i with a donor from the
                **farthest cluster** (cluster whose centroid is farthest from i), maximizing
                context shift.
              – data_type="spatial": kNN; replace with the **farthest** point (top 5% distances),
                not in the nearest-neighbor set.
              – data_type="temporal": windowing; replace with a row **window** steps away
                (default ≈10% of N), i.e., donor = (i + window) mod N.
            Meta → "context".
        • **A4 (benign noise)**: add small Gaussian noise per feature (no anomaly expected).
          Meta → None.

    The function balances A1–A4 across the selected rows (when feasible) and never samples more
    rows than available.  For small N/D we fall back gracefully (e.g., A2→A1 if D<2).

    Args
    ----
    X_test : (N, D) ndarray
        Test matrix to mutate (rows are records).
    partitions : ignored (kept for API compatibility).
    mutation_rate : float
        Fraction of rows to mutate (capped by N). If `ensure_all_types` and N≥4, we ensure each
        type appears at least once.
    seed : int
        RNG seed.
    a4_noise_std : float
        Std multiplier for A4 benign noise (small so A4 remains mostly non-anomalous).
    ensure_all_types : bool
        If True and N≥4, distribute mutations across A1..A4.
    data_type : {"tabular","spatial","temporal"}
        Strategy for **A3** (context) mutations.
    k_ctx : int
        #neighbors for spatial kNN.
    n_clusters : int or None
        #clusters for KMeans in tabular A3. Default ≈ sqrt(N).
    window : int or None
        Temporal offset for A3 windowing. Default ≈ 10% of N (at least 5).
    extreme_mult : float
        How far beyond the observed [lo, hi] we push A1/A2. New value = hi + m·(hi−lo) or
        lo − m·(hi−lo). Use 0.5–1.0 to make anomalies stronger.

    Returns
    -------
    X_mut : (N, D) ndarray
        Mutated copy of X_test.
    metas : dict[int, object]
        Mapping row index → expected explanation label:
          A1→(f,f), A2→(f1,f2), A3→"context", A4→None.
    """
    rng = np.random.default_rng(seed)
    X_mut = np.array(X_test, copy=True)
    N, D  = X_mut.shape

    if N == 0:
        return X_mut, {}

    # ----- how many rows to mutate (cap by N) -----
    requested = int(round(mutation_rate * N))
    if ensure_all_types and N >= 4:
        requested = max(requested, 4)     # ensure each type appears
    requested = max(1, min(requested, N)) # never exceed N

    # sample indices WITHOUT replacement
    sel = rng.choice(N, size=requested, replace=False)

    # ----- split selected rows across A1..A4 (balanced, with remainder) -----
    if ensure_all_types and N >= 4:
        base, rem = divmod(requested, 4)
        counts = [base + (i < rem) for i in range(4)]
        type_order = (["A1"] * counts[0] +
                      ["A2"] * counts[1] +
                      ["A3"] * counts[2] +
                      ["A4"] * counts[3])
    else:
        all_types = ["A1", "A2", "A3", "A4"]
        type_order = [all_types[i % 4] for i in range(requested)]

    # ---- helpers ----
    def _lo_hi_span(col: np.ndarray) -> tuple[float, float, float]:
        lo, hi = np.nanpercentile(col, [0.1, 99.9])  # tighter tails → stronger extremes
        if not np.isfinite(lo) or not np.isfinite(hi):
            lo, hi = float(np.nanmin(col)), float(np.nanmax(col))
        if lo == hi:
            hi = lo + 1e-6
        return lo, hi, (hi - lo)

    metas: dict[int, object] = {}

    # ---------- Precompute structures for A3 ----------
    mode = (data_type or "tabular").lower()
    if mode == "static":
        mode = "tabular"

    # TABULAR (KMeans across rows → far cluster donor)
    if mode == "tabular" and N >= 2:
        try:
            from sklearn.cluster import KMeans
            C = n_clusters if n_clusters is not None else int(max(2, min(N, round(np.sqrt(N)))))
            try:
                km = KMeans(n_clusters=C, random_state=seed, n_init="auto")
            except TypeError:
                km = KMeans(n_clusters=C, random_state=seed, n_init=10)
            labels = km.fit_predict(X_test)
            centroids = km.cluster_centers_   # (C,D)
            # members per cluster
            from collections import defaultdict
            cluster_members = defaultdict(list)
            for i, lab in enumerate(labels):
                cluster_members[int(lab)].append(i)
        except Exception:
            labels, centroids, cluster_members = None, None, None

    # SPATIAL (distance matrix / farthest donor)
    if mode == "spatial" and N >= 2:
        # full pairwise Euclidean distances (ok for typical N; for very large N replace with ANN)
        diffs = X_test[:, None, :] - X_test[None, :, :]
        distM = np.sqrt(np.sum(diffs * diffs, axis=2))  # (N,N)

    # TEMPORAL (window offset)
    if mode == "temporal":
        if window is None:
            window = max(5, int(round(0.10 * N)))  # ~10% of N
        window = int(max(1, min(window, max(1, N-1))))

    # ---------- apply mutations ----------
    for idx, mtype in zip(sel, type_order):
        idx = int(idx)

        if mtype == "A1":
            # push ONE feature beyond observed range by extreme_mult * span
            f = int(rng.integers(0, max(1, D)))
            lo, hi, span = _lo_hi_span(X_test[:, f])
            med = np.nanmedian(X_test[:, f])
            if X_test[idx, f] < med:
                new_val = hi + extreme_mult * span
            else:
                new_val = lo - extreme_mult * span
            jitter = float(rng.normal(0.0, 0.02 * span))
            X_mut[idx, f] = float(new_val + jitter)
            metas[idx] = (f, f)

        elif mtype == "A2":
            # pick two features; prefer different partitions if provided
            if D >= 2:
                if partitions and len(partitions) >= 2 and sum(len(p) for p in partitions) == D:
                    # sample two distinct partitions then a feature from each
                    p1, p2 = rng.choice(len(partitions), size=2, replace=False)
                    f1 = int(rng.choice(partitions[p1]))
                    f2 = int(rng.choice(partitions[p2]))
                else:
                    f1, f2 = rng.choice(D, size=2, replace=False)
                lo1, hi1, s1 = _lo_hi_span(X_test[:, f1])
                lo2, hi2, s2 = _lo_hi_span(X_test[:, f2])
                # push to opposite sides, beyond range
                X_mut[idx, f1] = hi1 + extreme_mult * s1
                X_mut[idx, f2] = lo2 - extreme_mult * s2
                metas[idx] = (int(f1), int(f2))
            else:
                # fallback to A1
                f = 0
                lo, hi, span = _lo_hi_span(X_test[:, f])
                X_mut[idx, f] = hi + extreme_mult * span
                metas[idx] = (f, f)

        elif mtype == "A3":
            if N < 2:
                # fallback to A4
                noise = rng.normal(0.0, a4_noise_std, size=D).astype(float)
                X_mut[idx] = X_mut[idx] + noise
                metas[idx] = None
                continue

            if mode == "tabular" and labels is not None and centroids is not None:
                lab_i = int(labels[idx])
                # pick the farthest centroid from x_i
                d2c = np.linalg.norm(centroids - X_test[idx], axis=1)
                far_lab = int(np.argmax(d2c))
                # choose donor from far_lab with largest distance to x_i
                cand = cluster_members[far_lab]
                if len(cand) == 0:
                    cand = [j for j in range(N) if j != idx]
                dists = np.linalg.norm(X_test[cand] - X_test[idx], axis=1)
                donor = int(cand[int(np.argmax(dists))])

            elif mode == "spatial":
                # farthest (top 5%) non-neighbor
                drow = distM[idx].copy()
                drow[idx] = -np.inf
                # pick from top-5% farthest
                k = max(1, int(round(0.05 * N)))
                farset = np.argsort(drow)[-k:]
                donor = int(rng.choice(farset))

            elif mode == "temporal":
                donor = int((idx + window) % N)

            else:
                # generic fallback: farthest by Euclidean distance
                drow = np.linalg.norm(X_test - X_test[idx], axis=1)
                drow[idx] = -np.inf
                donor = int(np.argmax(drow))

            X_mut[idx] = X_test[donor]
            metas[idx] = "context"

        elif mtype == "A4":
            # small benign noise (kept small so A4 remains mostly non-anomalous)
            noise = rng.normal(0.0, a4_noise_std, size=D).astype(float)
            X_mut[idx] = X_mut[idx] + noise
            metas[idx] = None

        else:
            # safety fallback → benign noise
            noise = rng.normal(0.0, a4_noise_std, size=D).astype(float)
            X_mut[idx] = X_mut[idx] + noise
            metas[idx] = None

    return X_mut, metas



# ==================== Evaluate Explanations on Mutations ====================
def evaluate_explanations_on_mutations(
    explanations: Dict[int, Dict],
    metas: Dict[int, Optional[Tuple[int,int]]],
    partitions: List[List[int]],
    topM: int = 3,                 # how many top disagreement pairs to consider
    use_violated_deps: bool = True,# also accept any pair in ex["violated_dependencies"]
    y_true: Optional[np.ndarray] = None,  # optional: if provided, p90 is computed on clean rows
    ref_max_dis_clean: Optional[np.ndarray] = None,  
) -> Dict[str, float]:
    """
    Mutation-based explanation evaluation (A1–A4), improved:

    • A1 (feature corruption): success if ANY of the top-M disagreement pairs
      includes the mutated partition k, OR if any violated dependency involves k.
    • A2 (cross-feature inconsistency): success if the unordered expected pair {k1,k2}
      is in the top-M disagreement pairs OR in violated dependencies.
    • A3 (context): success if ex['context_flag'] is True (or similarity < 0.9 fallback).
    • A4 (noise): success if no violated deps AND max_disagreement <= p90 of clean refs.

    Returns per-type accuracy and overall.
    """
    import numpy as np

    def _top_pairs_from_D(D: np.ndarray, M: int = 3) -> set:
        """Return set of up to M pairs (k,l) with largest D (k<l)."""
        if D is None:
            return set()
        K = D.shape[0]
        tri = np.triu(D, 1)
        # Handle pathological all-NaN/const
        if not np.isfinite(tri).any():
            return set()
        order = np.argsort(tri, axis=None)[::-1]  # descending by disagreement
        pairs = set()
        for idx in order:
            k, l = np.unravel_index(idx, (K, K))
            if k < l:
                pairs.add((int(k), int(l)))
                if len(pairs) >= M:
                    break
        return pairs

    f2p = _feature_to_partition_map(partitions)
    total = {"A1":0, "A2":0, "A3":0, "A4":0}
    correct = {"A1":0, "A2":0, "A3":0, "A4":0}

    # --- Reference p90 for A4 (prefer clean rows if y_true provided) ---
    if y_true is not None:
        clean_idxs = [i for i in explanations.keys() if y_true[i] == 0]
        ref_max_dis = [explanations[i]["max_disagreement"] for i in clean_idxs]
    else:
        # fallback: approximate clean as those with no dep violations & no context flag
        ref_max_dis = [
            ex["max_disagreement"]
            for ex in explanations.values()
            if not ex.get("violated_dependencies", []) and not ex.get("context_flag", False)
        ]


    # --- Reference p90 for A4, prefer truly clean distribution ---
    if ref_max_dis_clean is not None and len(ref_max_dis_clean) >= 20:
        p90 = float(np.quantile(ref_max_dis_clean, 0.90))
    elif y_true is not None:
        clean_idxs = [i for i in explanations.keys() if y_true[i] == 0]
        ref_max_dis = [explanations[i]["max_disagreement"] for i in clean_idxs]
        p90 = float(np.quantile(ref_max_dis, 0.90)) if len(ref_max_dis) >= 20 else 0.2
    else:
        ref_max_dis = [
            ex["max_disagreement"]
            for ex in explanations.values()
            if not ex.get("violated_dependencies", []) and not ex.get("context_flag", False)
        ]
        p90 = float(np.quantile(ref_max_dis, 0.90)) if len(ref_max_dis) >= 20 else 0.2

    for idx, expected in metas.items():
        ex = explanations.get(idx)
        if ex is None:
            continue

        # Build candidate pairs: top-M by D plus (optionally) violated deps
        D = ex.get("disagreement_matrix", None)
        cand_pairs = _top_pairs_from_D(D, M=topM)
        if use_violated_deps:
            viol = [tuple(sorted(p)) for p in ex.get("violated_dependencies", [])]
            cand_pairs |= set(viol)

        if expected == "context":
            total["A3"] += 1
            ok = bool(ex.get("context_flag", False))
            if not ok and ex.get("context_similarity") is not None:
                ok = float(ex["context_similarity"]) < 0.9
            correct["A3"] += int(ok)

        elif expected is None:
            total["A4"] += 1
            ok = (len(ex.get("violated_dependencies", [])) == 0) and (ex["max_disagreement"] <= p90)
            correct["A4"] += int(ok)

        else:
            # expected is (f1, f2)
            f1, f2 = expected
            k1, k2 = f2p[int(f1)], f2p[int(f2)]
            exp_pair = tuple(sorted((k1, k2)))

            if k1 == k2:
                # A1: success if any candidate pair mentions k1
                total["A1"] += 1
                ok = any((k1 == a or k1 == b) for (a, b) in cand_pairs)
                correct["A1"] += int(ok)
            else:
                # A2: success if expected pair is among candidates
                total["A2"] += 1
                ok = (exp_pair in cand_pairs)
                correct["A2"] += int(ok)

    # Accuracies
    accs = {}
    for t in ["A1","A2","A3","A4"]:
        accs[t] = (correct[t] / total[t]) if total[t] > 0 else np.nan
    tot = sum(total.values())
    cor = sum(correct.values())
    accs["overall"] = (cor / tot) if tot > 0 else np.nan
    return accs




In [8]:
def run_experiment(
    X_train,
    X_test,
    y_test,
    K=4,
    k_ctx=5,
    use_context=False,
    epochs=5,
    model_type="mlp",
    anom_percentile=95,
    pdf=None,
    data_type="tabular",
    mutate_fn=None,
):
    """
    End-to-end training + evaluation + mutation-based detection & explanation (CACL §5.2).

    What this does (merged pipeline):
      1) Train a MultiView contrastive model (optionally context-aware).
      2) Evaluate anomaly detection on the unmutated TEST set (y_test).
      3) Perform mutation-based evaluation:
         • Create synthetic anomalies A1–A4 on X_test (via mutate_fn or mutate_test_set).
         • Compute explanations for each mutated record (disagreement, deps, context).
         • Compute detection scores (global+context+disagreement blend) and robust threshold.
         • Report per-type detection metrics (A1–A4) + A4 FPR and explanation accuracy.

    Notes:
      • Uses robust threshold selection to avoid "all-zeros" predictions.
      • Context embeddings are computed once from the unmutated X_test and reused.
      • No duplicated helpers (_cos, _avg_partition_embeddings, thresholding).

    Returns dict including:
      - Standard detection metrics on TEST (ROC/PR/F1/Precision/Recall/Accuracy)
      - Mutation-based explanation accuracy per type
      - Mutation-based per-type detection metrics (+ A4 FPR)
      - Scores, preds, threshold, training losses, etc.
    """
    # -------------------- imports --------------------
    import os, time
    import numpy as np
    import matplotlib.pyplot as plt
    import torch
    import torch.nn.functional as F
    from torch.utils.data import DataLoader
    from sklearn.metrics import (
        roc_auc_score, average_precision_score, f1_score,
        precision_score, recall_score, accuracy_score,
        roc_curve, precision_recall_curve
    )

    # -------------------- device ---------------------
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # -------------------- helpers --------------------
    def _cos(a, b):
        a = np.asarray(a, dtype=float); b = np.asarray(b, dtype=float)
        a = a / (np.linalg.norm(a) + 1e-12)
        b = b / (np.linalg.norm(b) + 1e-12)
        return float(np.dot(a, b))

    def _to1d(x):
        if hasattr(x, "detach"):
            x = x.detach().cpu().numpy()
        return np.asarray(x, dtype=float).ravel()

    def _pad_or_truncate(vec, d):
        v = _to1d(vec)
        if v.shape[0] == d:
            return v
        out = np.zeros(d, dtype=float)
        n = min(d, v.shape[0])
        out[:n] = v[:n]
        return out

    def _avg_partition_embeddings(model, X, partitions):
        """
        Average partition embeddings per record (N, d).
        Works regardless of whether model exposes encode_partitions / encode_partition or only forward.
        """
        N = X.shape[0]
        Kp = len(partitions)

        # Fast path: model.encode_partitions
        if hasattr(model, "encode_partitions"):
            out = model.encode_partitions(X, partitions)
            if isinstance(out, (list, tuple)) and len(out) > 0:
                stacks = [np.asarray(o) for o in out]             # list[K](N,d)
                return np.mean(np.stack(stacks, axis=0), axis=0)  # (N,d)
            if hasattr(out, "detach") and getattr(out, "ndim", 0) == 3:
                return out.detach().cpu().numpy().mean(axis=1)    # (N,K,d)->(N,d)
            out = np.asarray(out)
            if out.ndim == 3 and out.shape[1] == Kp:
                return out.mean(axis=1)

        # encode_partition path
        if hasattr(model, "encode_partition"):
            d = _to1d(model.encode_partition(_to1d(X[0, partitions[0]]), 0)).shape[0]
            E = np.zeros((N, Kp, d), dtype=float)
            for i in range(N):
                for k, part in enumerate(partitions):
                    vec = model.encode_partition(_to1d(X[i, part]), k)
                    E[i, k] = _pad_or_truncate(vec, d)
            return E.mean(axis=1)

        # fallback: run forward
        with torch.no_grad():
            E = []
            for i in range(N):
                views = [torch.tensor(X[i, p], dtype=torch.float32, device=device).unsqueeze(0)
                         for p in partitions]
                out = model(views)                                 # list[K](1,d)
                z   = torch.stack([o[0] for o in out], dim=0)      # (K,d)
                E.append(z.mean(dim=0).cpu().numpy())              # (d,)
        return np.vstack(E)                                        # (N,d)

    def _get_partition_embeddings(model, X, partitions, i):
        """
        Return list[K] of (d,) embeddings for record i (for explain_record).
        """
        with torch.no_grad():
            views = [torch.tensor(X[i, p], dtype=torch.float32, device=device).unsqueeze(0)
                     for p in partitions]
            out = model(views)                 # list[K](1,d)
        return [out[k][0].detach().cpu().numpy() for k in range(len(partitions))]

    def _pick_threshold(scores: np.ndarray, y_true: np.ndarray, normal_percentile: float = 95.0):
        """
        Robust thresholding: pick percentile on NORMALS; if degenerate (all-0/1), fall back
        to F1-optimal threshold searched over quantiles.
        """
        scores = np.asarray(scores, dtype=float)
        y_true = np.asarray(y_true, dtype=int)
        neg = (y_true == 0)

        if neg.any():
            thr = float(np.nanpercentile(scores[neg], normal_percentile))
        else:
            thr = float(np.nanpercentile(scores, normal_percentile))
        preds = (scores >= thr).astype(int)

        if preds.sum() == 0 or preds.sum() == len(preds):
            qs = np.linspace(50, 99.5, 100)
            best_f1, best_thr = -1.0, thr
            for q in qs:
                t = float(np.nanpercentile(scores, q))
                p = (scores >= t).astype(int)
                tp = int(((p == 1) & (y_true == 1)).sum())
                fp = int(((p == 1) & (y_true == 0)).sum())
                fn = int(((p == 0) & (y_true == 1)).sum())
                prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
                rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
                f1   = (2*prec*rec)/(prec+rec) if (prec+rec) > 0 else 0.0
                if f1 > best_f1:
                    best_f1, best_thr = f1, t
            thr = best_thr
            preds = (scores >= thr).astype(int)
        return thr, preds

    print("=" * 80)
    print(f"[Experiment] use_context={use_context}, K={K}, epochs={epochs}, model={model_type}")
    print("=" * 80)
    start_time = time.time()

    # -------------------- partitions & context (train) --------------------
    partitions = create_feature_partitions(X_train, K)
    try:
        ctx_idx_train = build_context_groups(X_train, k=k_ctx, mode=data_type)
    except TypeError:
        ctx_idx_train = build_context_groups(X_train, k=k_ctx)

    # -------------------- model --------------------
    model = MultiViewModel(partitions, encoder_type=model_type).to(device)
    model.device = device
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    dataset = PartitionedDataset(X_train, partitions)
    loader  = DataLoader(dataset, batch_size=128, shuffle=True, collate_fn=collate_fn)

    # -------------------- train --------------------
    losses = []
    model.train()
    for epoch in range(epochs):
        epoch_losses = []
        for views, batch_idx in loader:
            views = [v.to(device) for v in views]
            embs = model(views)
            loss = contrastive_loss_ctx(embs, ctx_idx_train, batch_idx) if use_context else contrastive_loss(embs)
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            epoch_losses.append(float(loss.item()))
        avg_loss = float(np.mean(epoch_losses)) if epoch_losses else 0.0
        losses.append(avg_loss)
        print(f"Epoch {epoch+1}/{epochs} - Avg Loss: {avg_loss:.4f}")

    duration = time.time() - start_time
    print(f"\n[Training Complete] Duration: {duration:.2f} seconds")
    print("[Evaluating on Test Set]...")

    # -------------------- test-set detection (unmutated, y_test) --------------------
    model.eval()
    H_avg_test = _avg_partition_embeddings(model, X_test, partitions)       # (N,d)

    # global-center score
    center_vec    = H_avg_test.mean(axis=0, keepdims=True)
    center_scores = np.array([1.0 - _cos(H_avg_test[i], center_vec[0]) for i in range(len(H_avg_test))], dtype=float)

    # context groups & embeddings from UNMUTATED test
    try:
        ctx_idx_test = build_context_groups(X_test, k=k_ctx, mode=data_type)
    except TypeError:
        ctx_idx_test = build_context_groups(X_test, k=k_ctx)
    ctx_avg_test = H_avg_test

    # context score: 1 - mean cosine to neighbors
    ctx_scores = np.zeros(len(H_avg_test), dtype=float)
    for i, neigh in enumerate(ctx_idx_test):
        if not neigh: 
            ctx_scores[i] = 0.0
            continue
        sims = [_cos(H_avg_test[i], ctx_avg_test[j]) for j in neigh if j < len(ctx_avg_test)]
        ctx_scores[i] = 1.0 - (np.mean(sims) if sims else 1.0)

    # blend (global + context); small dis term can be added here too; keep simple for vanilla test
    alpha_ctx  = 0.30
    all_scores = (1.0 - alpha_ctx) * center_scores + alpha_ctx * ctx_scores

    true_labels = np.asarray(y_test, dtype=int)
    thr_test, preds_test = _pick_threshold(all_scores, true_labels, normal_percentile=anom_percentile)

    roc  = roc_auc_score(true_labels, all_scores)
    pr   = average_precision_score(true_labels, all_scores)
    f1   = f1_score(true_labels, preds_test)
    prec = precision_score(true_labels, preds_test)
    rec  = recall_score(true_labels, preds_test)
    acc  = accuracy_score(true_labels, preds_test)

    print("\n[Evaluation Metrics — Test set]")
    print(f"ROC-AUC: {roc:.4f} | PR-AUC: {pr:.4f}")
    print(f"F1: {f1:.4f} | Precision: {prec:.4f} | Recall: {rec:.4f} | Accuracy: {acc:.4f}")
    print(f"Threshold: {thr_test:.4f}")

    # -------------------- model save --------------------
    model_dir = "models"
    os.makedirs(model_dir, exist_ok=True)
    model_path = os.path.join(model_dir, f"model_K{K}_ctx{use_context}_arch{model_type}.pt")
    torch.save(model.state_dict(), model_path)
    print(f"[Model Saved] → {model_path}\n")

    # -------------------- plots to PDF (test-set) --------------------
    if pdf is not None:
        fig = plot_training_loss(losses, title=f"Training Loss (K={K}, Context-Aware={use_context})")
        pdf.savefig(fig); plt.close(fig)

        plt.figure(figsize=(7, 4))
        plt.hist(all_scores[true_labels == 0], bins=40, alpha=0.6, label='Normal', color='lightgray', edgecolor='k', hatch='//')
        plt.hist(all_scores[true_labels == 1], bins=40, alpha=0.6, label='Anomalous', color='gray', edgecolor='k', hatch='xx')
        plt.axvline(thr_test, color='blue', linestyle='--', label=f'Percentile {anom_percentile}')
        plt.title(f"Anomaly Score Distribution (K={K}, Context={use_context})")
        plt.xlabel("Anomaly Score"); plt.ylabel("Frequency")
        plt.legend(); plt.grid(True); plt.tight_layout()
        pdf.savefig(); plt.close()

        fpr, tpr, _ = roc_curve(true_labels, all_scores)
        prec_c, rec_c, _ = precision_recall_curve(true_labels, all_scores)

        plt.figure(); plt.plot(fpr, tpr, label=f"ROC (AUC = {roc:.4f})")
        plt.plot([0, 1], [0, 1], linestyle='--', color='gray')
        plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
        plt.title("ROC Curve"); plt.legend(); plt.grid(True)
        pdf.savefig(); plt.close()

        plt.figure(); plt.plot(rec_c, prec_c, label=f"PR (AUC = {pr:.4f})")
        plt.xlabel("Recall"); plt.ylabel("Precision")
        plt.title("Precision-Recall Curve"); plt.legend(); plt.grid(True)
        pdf.savefig(); plt.close()

        summary_data = [
            ["Data Type", data_type],
            ["K (Partitions)", str(K)],
            ["k (Context Groups)", str(k_ctx)],
            ["Context-Aware", str(use_context)],
            ["Epochs", str(epochs)],
            ["Model", model_type],
            ["ROC-AUC", f"{roc:.4f}"],
            ["PR-AUC", f"{pr:.4f}"],
            ["F1 Score", f"{f1:.4f}"],
            ["Precision", f"{prec:.4f}"],
            ["Recall", f"{rec:.4f}"],
            ["Accuracy", f"{acc:.4f}"],
            ["Threshold", f"{thr_test:.4f}"],
            ["Duration (sec)", f"{duration:.2f}"],
        ]
        fig, ax = plt.subplots(figsize=(8.5, 5))
        ax.axis('off')
        table = plt.table(cellText=summary_data, colLabels=["Metric", "Value"], loc='center', cellLoc='left')
        table.auto_set_font_size(False); table.set_fontsize(10); table.scale(1.1, 1.3)
        pdf.savefig(fig); plt.close(fig)

    # -------------------- mutation-based evaluation (detection + explanation) --------------------
    # context groups/embeddings from UNMUTATED test (reuse above)
    ctx_groups_unmut = ctx_idx_test
    ctx_emb_unmut    = ctx_avg_test

    if mutate_fn is None:
        mutate_fn = globals().get("mutate_test_set", None)
        if mutate_fn is None:
            raise ValueError("Please provide mutate_fn or define mutate_test_set().")
    try:
        X_mut, metas = mutate_fn(X_test.copy(), partitions, data_type=data_type)
    except TypeError:
        X_mut, metas = mutate_fn(X_test.copy(), partitions)

    # binary labels for mutation set: 1 for A1/A2/A3, 0 for A4 (benign)
    y_mut = np.zeros(len(X_test), dtype=int)
    for i, meta in metas.items():
        y_mut[i] = 0 if (meta is None) else 1

    # dependency matrix from TRAIN inliers
    dependency_matrix = compute_dependency_matrix(X_train, model, partitions, eta=0.2)

    # Build explanations on UNMUTATED X_test to obtain a clean reference distribution
    explanations_unmut = {}
    for i in range(len(X_test)):
        rec_embs = _get_partition_embeddings(model, X_test, partitions, i)
        explanations_unmut[i] = explain_record(
            record_embeddings=rec_embs,
            dependency_matrix=dependency_matrix,
            eta=0.2,  # keep in sync with compute_dependency_matrix or your relaxed value
        )

    # Prefer truly clean rows via y_test==0 if labels exist
    ref_clean_max_dis = [
        explanations_unmut[i]["max_disagreement"]
        for i in range(len(X_test)) if (y_test[i] == 0)
    ]
    ref_clean_max_dis = np.asarray(ref_clean_max_dis, dtype=float)


    # calibration from "inliers" in mutation set
    cal_idx = np.where(y_mut == 0)[0]
    calibration_inliers_idx = cal_idx[:min(200, len(cal_idx))].tolist()

    # embeddings for mutated set
    H_avg_mut = _avg_partition_embeddings(model, X_mut, partitions)

    # context calibration (quantiles) using UNMUTATED context embeddings
    ref_ctx_sim = None
    if ctx_groups_unmut is not None and calibration_inliers_idx:
        sims = []
        for i in calibration_inliers_idx:
            if i >= len(ctx_groups_unmut): 
                continue
            neigh = ctx_groups_unmut[i]
            if not neigh:
                continue
            s = [_cos(ctx_emb_unmut[i], ctx_emb_unmut[j]) for j in neigh if j < len(ctx_emb_unmut)]
            if s:
                sims.append(np.mean(s))
        if sims:
            ref_ctx_sim = np.asarray(sims, dtype=float)

    # explanations for mutated set
    explanations = {}
    for i in range(len(X_mut)):
        rec_embs = _get_partition_embeddings(model, X_mut, partitions, i)
        ex = explain_record(
            record_embeddings=rec_embs,
            dependency_matrix=dependency_matrix,
            eta=0.2,
            context_avg_embeddings=ctx_emb_unmut,
            context_indices=ctx_groups_unmut[i] if (ctx_groups_unmut and i < len(ctx_groups_unmut)) else None,
            context_threshold_quantile=0.1,
            ref_ctx_sim_distribution=ref_ctx_sim,
        )
        explanations[i] = ex

    # explanation accuracy per type
    explanation_accuracy = evaluate_explanations_on_mutations(
        explanations=explanations,
        metas=metas,
        partitions=partitions,
        y_true=y_mut,
        ref_max_dis_clean=ref_clean_max_dis  # <-- NEW
    )

    # mutation detection scores: blend global + context + disagreement (stronger)
    center_vec_mut = H_avg_mut.mean(axis=0, keepdims=True)
    center_scores_mut = np.array([1.0 - _cos(H_avg_mut[i], center_vec_mut[0]) for i in range(len(H_avg_mut))], dtype=float)

    # context inconsistency wrt UNMUTATED neighbors
    ctx_scores_mut = np.zeros(len(H_avg_mut), dtype=float)
    if (ctx_groups_unmut is not None) and (ctx_emb_unmut is not None):
        for i in range(len(H_avg_mut)):
            neigh = ctx_groups_unmut[i] if i < len(ctx_groups_unmut) else []
            if not neigh:
                ctx_scores_mut[i] = 0.0
                continue
            sims = [_cos(H_avg_mut[i], ctx_emb_unmut[j]) for j in neigh if j < len(ctx_emb_unmut)]
            ctx_scores_mut[i] = 1.0 - (np.mean(sims) if sims else 1.0)

    # disagreement term: max upper-triangle of D from explanations
    max_dis_list = np.array([
        float(np.nanmax(np.triu(explanations[i]["disagreement_matrix"], 1)))
        for i in range(len(H_avg_mut))
    ], dtype=float)

    w_global, w_ctx, w_dis = 0.6, 0.3, 0.1
    s = w_global + w_ctx + w_dis
    w_global, w_ctx, w_dis = w_global/s, w_ctx/s, w_dis/s
    scores_mut = w_global * center_scores_mut + w_ctx * ctx_scores_mut + w_dis * max_dis_list

    thr_mut, preds_mut = _pick_threshold(scores_mut, y_mut, normal_percentile=anom_percentile)

    roc_m  = roc_auc_score(y_mut, scores_mut)
    pr_m   = average_precision_score(y_mut, scores_mut)
    f1_m   = f1_score(y_mut, preds_mut)
    prec_m = precision_score(y_mut, preds_mut)
    rec_m  = recall_score(y_mut, preds_mut)
    acc_m  = accuracy_score(y_mut, preds_mut)

    # per-type detection metrics + A4 FPR
    from collections import defaultdict
    type_labels = {}
    for idx, meta in (metas or {}).items():
        if meta is None:
            type_labels[idx] = "A4"
        elif meta == "context":
            type_labels[idx] = "A3"
        elif isinstance(meta, tuple) and meta[0] == meta[1]:
            type_labels[idx] = "A1"
        else:
            type_labels[idx] = "A2"

    type_to_indices = defaultdict(list)
    for i, t in type_labels.items():
        type_to_indices[t].append(int(i))

    nonmut = np.where(y_mut == 0)[0]
    detection_accuracy_by_type = {}
    for t in ["A1", "A2", "A3", "A4"]:
        pos = np.array(type_to_indices.get(t, []), dtype=int)
        if pos.size == 0 or nonmut.size == 0:
            detection_accuracy_by_type[t] = {
                "roc_auc": np.nan, "pr_auc": np.nan, "f1": np.nan,
                "precision": np.nan, "recall": np.nan, "accuracy": np.nan
            }
            continue
        idxs = np.concatenate([pos, nonmut])
        y_t = y_mut[idxs]; s_t = scores_mut[idxs]; p_t = preds_mut[idxs]
        detection_accuracy_by_type[t] = {
            "roc_auc": roc_auc_score(y_t, s_t) if len(np.unique(y_t)) > 1 else np.nan,
            "pr_auc":  average_precision_score(y_t, s_t) if y_t.sum() > 0 else np.nan,
            "f1":      f1_score(y_t, p_t),
            "precision": precision_score(y_t, p_t, zero_division=0),
            "recall":    recall_score(y_t, p_t, zero_division=0),
            "accuracy":  accuracy_score(y_t, p_t),
        }

    a4_idx = np.array(type_to_indices.get("A4", []), dtype=int)
    a4_fpr = float(preds_mut[a4_idx].mean()) if a4_idx.size > 0 else np.nan
    if "A4" not in detection_accuracy_by_type:
        detection_accuracy_by_type["A4"] = {}
    detection_accuracy_by_type["A4"]["fpr"] = a4_fpr

    # -------------------- PDF tables (mutation eval) --------------------
    if pdf is not None:
        # Explanation accuracy
        row_map = [("A1","A1_explanation_accuracy"),
                   ("A2","A2_explanation_accuracy"),
                   ("A3","A3_explanation_accuracy"),
                   ("A4","A4_explanation_accuracy")]
        table_rows = []
        for key, label in row_map:
            val = explanation_accuracy.get(key, np.nan)
            table_rows.append([label, f"{val:.2f}" if np.isfinite(val) else "N/A"])
        fig, ax = plt.subplots(figsize=(7.5, 2.6)); ax.axis('off')
        t = plt.table(cellText=table_rows, colLabels=["Anomaly Type", "Explanation Accuracy"],
                      loc='center', cellLoc='center')
        t.auto_set_font_size(False); t.set_fontsize(10); t.scale(1.15, 1.25)
        pdf.savefig(fig); plt.close(fig)

        # Per-type detection metrics (compact + full)
        metric_names = ["f1", "roc_auc", "pr_auc", "precision", "recall", "accuracy"]
        det_rows = []
        for tname in ["A1","A2","A3","A4"]:
            d = detection_accuracy_by_type.get(tname, {})
            for m in metric_names:
                val = d.get(m, np.nan)
                det_rows.append([f"{tname}_detection_{m.upper()}",
                                 f"{val:.2f}" if np.isfinite(val) else "N/A"])
            if tname == "A4":
                val = d.get("fpr", np.nan)
                det_rows.append([f"{tname}_detection_FPR",
                                 f"{val:.2f}" if np.isfinite(val) else "N/A"])
        fig, ax = plt.subplots(figsize=(7.5, 5.0)); ax.axis('off')
        t = plt.table(cellText=det_rows, colLabels=["Mutation Type", "Metric Value"],
                      loc='center', cellLoc='center')
        t.auto_set_font_size(False); t.set_fontsize(10); t.scale(1.1, 1.3)
        pdf.savefig(fig); plt.close(fig)

    # -------------------- return --------------------
    return {
        # standard test-set detection
        'preds': preds_test,
        'roc_auc': roc,
        'pr_auc': pr,
        'f1': f1,
        'precision': prec,
        'recall': rec,
        'accuracy': acc,
        'threshold': thr_test,
        'losses': losses,
        'duration_sec': duration,
        'K': K,
        'use_context': use_context,

        # mutation-based results
        'mutation': {
            'scores': scores_mut,
            'preds': preds_mut,
            'y_true': y_mut,
            'threshold': thr_mut,
            'roc_auc': roc_m,
            'pr_auc': pr_m,
            'f1': f1_m,
            'precision': prec_m,
            'recall': rec_m,
            'accuracy': acc_m,
            'explanation_accuracy': explanation_accuracy,
            'detection_accuracy_by_type': detection_accuracy_by_type,
            'a4_false_positive_rate': a4_fpr,
        }
    }


In [9]:

# =========================================
# STEP 6: Run All Experiments
# =========================================
from matplotlib.backends.backend_pdf import PdfPages

# === MLP Encoder===
pdf_file = PdfPages("/home/azureuser/cloudfiles/code/Users/hhomayouni/context-aware contrastive learning for anomaly detection and explanation/results/BC.pdf")
print("\n[Exp 1] Context-Aware Contrastive | K > 2")
results1 = run_experiment(X_train, X_test, y_test, K=4, use_context=True,model_type="mlp", anom_percentile=100-anom_threshold,pdf=pdf_file, data_type="tabular")

print("\n[Exp 2] Context-Aware Contrastive | K = 2")
results2 = run_experiment(X_train, X_test, y_test, K=2, use_context=True, model_type="mlp", anom_percentile=100-anom_threshold,pdf=pdf_file, data_type="tabular")

print("\n[Exp 3] Regular Contrastive | K > 2")
results3 = run_experiment(X_train, X_test, y_test, K=4, use_context=False,model_type="mlp", anom_percentile=100-anom_threshold,pdf=pdf_file, data_type="tabular")

print("\n[Exp 4] Regular Contrastive | K = 2")
results4 = run_experiment(X_train, X_test, y_test, K=2, use_context=False, model_type="mlp", anom_percentile=100-anom_threshold,pdf=pdf_file, data_type="tabular")



# === Transformer Encoder===
print("\n[Exp 1] Context-Aware Contrastive | K > 2")
results1 = run_experiment(X_train, X_test, y_test, K=4, use_context=True,model_type="transformer", anom_percentile=100-anom_threshold,pdf=pdf_file, data_type="tabular",mutate_fn=mutate_test_set)

print("\n[Exp 2] Context-Aware Contrastive | K = 2")
results2 = run_experiment(X_train, X_test, y_test, K=2, use_context=True, model_type="transformer", anom_percentile=100-anom_threshold,pdf=pdf_file, data_type="tabular")

print("\n[Exp 3] Regular Contrastive | K > 2")
results3 = run_experiment(X_train, X_test, y_test, K=4, use_context=False,model_type="transformer", anom_percentile=100-anom_threshold,pdf=pdf_file, data_type="tabular")

print("\n[Exp 4] Regular Contrastive | K = 2")
results4 = run_experiment(X_train, X_test, y_test, K=2, use_context=False, model_type="transformer", anom_percentile=100-anom_threshold,pdf=pdf_file, data_type="tabular")

pdf_file.close()



/anaconda/envs/azureml_py38/lib/python3.10/site-packages/torch/nn/modules/transformer.py:307: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")


Epoch 1/5 - Avg Loss: 5.1243
Epoch 2/5 - Avg Loss: 4.5033
Epoch 3/5 - Avg Loss: 4.4578
Epoch 4/5 - Avg Loss: 4.4142
Epoch 5/5 - Avg Loss: 4.4012

[Training Complete] Duration: 1515.82 seconds
[Evaluating on Test Set]...

[Evaluation Metrics — Test set]
ROC-AUC: 0.9244 | PR-AUC: 0.9181
F1: 0.8632 | Precision: 0.8723 | Recall: 0.8542 | Accuracy: 0.9051
Threshold: 0.0519
[Model Saved] → models/model_K4_ctxTrue_archtransformer.pt

